# A04 house_price — torch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from assignment_runtime import enable_training_output_overlay
    enable_training_output_overlay()
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT = Path(_migration['logical_root'])


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Housing Price: dùng đặc trưng property để dự đoán giá nhà liên tục. A04 dùng King County House Sales trong `datasets/housing_price/a04`, tập trung vào deep learning regression trên dữ liệu tabular và so sánh cẩn thận với A03.


In [ ]:
if RUN_TRAINING:
    from pathlib import Path
    import json
    import pandas as pd

    def find_project_root(start=None):
        start = Path.cwd() if start is None else Path(start)
        for path in [start.resolve(), *start.resolve().parents]:
            if (path / "datasets" / "housing_price" / "a04").exists() and (path / "pipeline").exists():
                return path
        raise FileNotFoundError("Cannot locate project root from current working directory.")

    PROJECT_ROOT = find_project_root()
    DATA_DIR = PROJECT_ROOT / "datasets" / "housing_price" / "a04"
    A03_NOTEBOOK = PROJECT_ROOT / "pipeline" / "house_price_pipeline.ipynb"
    A03_METADATA_PATH = PROJECT_ROOT / "pipeline" / "house_price" / "metadata.json"
    A03_COMPARISON_PATH = PROJECT_ROOT / "pipeline" / "house_price" / "final_comparison.csv"
    A04_METADATA_PATH = DATA_DIR / "metadata.json"

    with A04_METADATA_PATH.open("r", encoding="utf-8") as f:
        dataset_metadata = json.load(f)

    overview = {
        "A03 task": "Regression: housing/property features -> house price",
        "A03 target": "Price_BillionVND",
        "A03 notebook exists": A03_NOTEBOOK.exists(),
        "A04 dataset": dataset_metadata["dataset_name"],
        "A04 source": dataset_metadata["source"],
        "A04 target": dataset_metadata["target"],
        "A04 samples": dataset_metadata["samples"],
        "A04 features": dataset_metadata["features"],
        "Task type": "Regression",
    }
    pd.DataFrame([overview]).T.rename(columns={0: "Value"})


Output cho thấy điều gì?

Bảng xác nhận A04 giữ đúng bài toán regression của A03: đầu vào là feature mô tả nhà/property, target là giá bán liên tục. Dataset A04 đổi sang King County nên metric cuối chỉ so sánh workflow và chất lượng dữ liệu, không dùng như benchmark trực tiếp với A03.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để path portable trong repo.


In [ ]:
import os
import random
import time
import warnings
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
ARTIFACT_DIR = PROJECT_ROOT / 'pipeline04' / 'house_price'
MODEL_DIR = ARTIFACT_DIR / 'models'
METRIC_DIR = ARTIFACT_DIR / 'metrics'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'PyTorch': torch.__version__, 'PyTorch device': str(DEVICE), 'Project root is portable': PROJECT_ROOT.name == 'assign'}
pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


Output cho thấy điều gì?

Environment có đủ hai framework để chạy cùng một dữ liệu đã preprocess. CPU vẫn phù hợp vì King County có khoảng vài chục nghìn dòng, nhỏ hơn nhiều so với các bài toán ảnh/video.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/housing_price/a04`. File gốc được giữ nguyên; mọi xử lý nằm trong DataFrame và pipeline preprocessing.


In [ ]:
if RUN_TRAINING:
    csv_files = sorted(DATA_DIR.glob("*.csv"))
    raw_files = [
        {
            "file": path.relative_to(PROJECT_ROOT).as_posix(),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
        }
        for path in csv_files
    ]
    display(pd.DataFrame(raw_files))

    DATA_PATH = DATA_DIR / "kc_house_data.csv"
    TARGET = "price"
    df_raw = pd.read_csv(DATA_PATH)

    print("Loaded dataset:", DATA_PATH.relative_to(PROJECT_ROOT).as_posix())
    print("Shape:", df_raw.shape)
    display(df_raw.head())


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. `price` là target regression liên tục; các cột còn lại mô tả thời gian bán, diện tích, vị trí, tình trạng và đặc điểm property.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, phân phối target, descriptive statistics và các giá trị housing bất thường.


In [ ]:
if RUN_TRAINING:
    shape_info = pd.DataFrame(
        {
            "Metric": ["Rows", "Columns", "Candidate feature columns"],
            "Value": [df_raw.shape[0], df_raw.shape[1], df_raw.shape[1] - 1],
        }
    )
    dtype_info = df_raw.dtypes.astype(str).reset_index()
    dtype_info.columns = ["Column", "Dtype"]

    missing = df_raw.isna().sum().reset_index()
    missing.columns = ["Column", "MissingCount"]
    missing["MissingRate"] = missing["MissingCount"] / len(df_raw)
    duplicate_count = int(df_raw.duplicated().sum())
    property_duplicate_count = int(df_raw['id'].duplicated().sum()) if 'id' in df_raw.columns else 0

    price_stats = df_raw[TARGET].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame("price").T
    price_skew = float(df_raw[TARGET].skew())
    q1, q3 = df_raw[TARGET].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_upper = q3 + 1.5 * iqr
    price_outliers = int((df_raw[TARGET] > outlier_upper).sum())

    quality_checks = pd.DataFrame(
        {
            "Check": [
                "price <= 0",
                "sqft_living <= 0",
                "sqft_lot <= 0",
                "bedrooms == 0",
                "bedrooms > 15",
                "bathrooms == 0",
                "yr_built <= 0",
            ],
            "Count": [
                int((df_raw[TARGET] <= 0).sum()),
                int((df_raw["sqft_living"] <= 0).sum()),
                int((df_raw["sqft_lot"] <= 0).sum()),
                int((df_raw["bedrooms"] == 0).sum()),
                int((df_raw["bedrooms"] > 15).sum()),
                int((df_raw["bathrooms"] == 0).sum()),
                int((df_raw["yr_built"] <= 0).sum()),
            ],
        }
    )

    display(shape_info)
    display(dtype_info)
    display(missing[missing["MissingCount"] > 0] if missing["MissingCount"].sum() else pd.DataFrame({"Missing": ["No missing values detected"]}))
    print("Duplicate rows:", duplicate_count)
    print("Repeated property id rows:", property_duplicate_count)
    print("Price skew:", round(price_skew, 4))
    print("IQR upper outlier boundary:", round(float(outlier_upper), 2))
    print("Price IQR outlier count:", price_outliers)
    display(price_stats.round(2))
    display(df_raw.describe().T.round(2))
    display(quality_checks)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(df_raw[TARGET], bins=60, color="#4C78A8")
    axes[0].set_title("Price Distribution")
    axes[0].set_xlabel("price")
    axes[0].set_ylabel("count")
    axes[1].hist(np.log1p(df_raw[TARGET]), bins=60, color="#F58518")
    axes[1].set_title("log1p(price) Distribution")
    axes[1].set_xlabel("log1p(price)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "target_price_distribution.png", dpi=160)
    plt.show()


Output cho thấy điều gì?

Dataset gốc không có missing value và không có full duplicate rows, nhưng có repeated property `id` do một số nhà được bán nhiều lần. Đây là rủi ro leakage nếu cùng một property xuất hiện ở nhiều split, nên bước chuẩn bị sẽ giữ latest sale per `id` trước khi chia train/validation/test. Phân phối `price` lệch phải rõ rệt, vì vậy notebook dùng log1p target cho training và inverse transform khi báo metrics trên price scale gốc.


# 5. Data Preparation

Cell này làm gì?

Tạo feature engineering từ `date`, loại identifier `id`, chia train/validation/test theo price bins, fit preprocessing chỉ trên train, và quyết định target log-transform dựa trên skew.


In [ ]:
if RUN_TRAINING:
    def add_housing_features(df):
        data = df.copy()
        sale_date = pd.to_datetime(data["date"].astype(str).str.slice(0, 8), format="%Y%m%d", errors="coerce")
        data["sale_year"] = sale_date.dt.year
        data["sale_month"] = sale_date.dt.month
        data["house_age_at_sale"] = (data["sale_year"] - data["yr_built"]).clip(lower=0)
        data["renovated_flag"] = (data["yr_renovated"] > 0).astype(int)
        data["years_since_renovation"] = np.where(
            data["yr_renovated"] > 0,
            (data["sale_year"] - data["yr_renovated"]).clip(lower=0),
            0,
        )
        data["basement_flag"] = (data["sqft_basement"] > 0).astype(int)
        data["zipcode"] = data["zipcode"].astype(str)
        return data

    df_model = df_raw.drop_duplicates().copy()
    repeated_property_rows_removed = int(df_model['id'].duplicated().sum())
    df_model = df_model.sort_values('date').drop_duplicates(subset=['id'], keep='last')
    df_model = df_model[df_model[TARGET] > 0].reset_index(drop=True)
    df_model = add_housing_features(df_model)

    numeric_features = [
        "bedrooms",
        "bathrooms",
        "sqft_living",
        "sqft_lot",
        "floors",
        "waterfront",
        "view",
        "condition",
        "grade",
        "sqft_above",
        "sqft_basement",
        "yr_built",
        "yr_renovated",
        "lat",
        "long",
        "sqft_living15",
        "sqft_lot15",
        "sale_year",
        "sale_month",
        "house_age_at_sale",
        "renovated_flag",
        "years_since_renovation",
        "basement_flag",
    ]
    categorical_features = ["zipcode"]
    feature_cols = numeric_features + categorical_features

    X = df_model[feature_cols]
    y = df_model[TARGET].astype(float)
    price_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

    X_train_raw, X_temp_raw, y_train, y_temp, bins_train, bins_temp = train_test_split(
        X,
        y,
        price_bins,
        test_size=0.30,
        random_state=SEED,
        stratify=price_bins,
    )
    X_valid_raw, X_test_raw, y_valid, y_test = train_test_split(
        X_temp_raw,
        y_temp,
        test_size=0.50,
        random_state=SEED,
        stratify=bins_temp,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_features,
            ),
        ],
        sparse_threshold=0.0,
    )

    X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
    X_valid = preprocessor.transform(X_valid_raw).astype("float32")
    X_test = preprocessor.transform(X_test_raw).astype("float32")
    joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

    y_train_np = y_train.to_numpy(dtype="float32")
    y_valid_np = y_valid.to_numpy(dtype="float32")
    y_test_np = y_test.to_numpy(dtype="float32")
    target_skew_train = float(pd.Series(y_train_np).skew())
    USE_LOG_TARGET = target_skew_train > 1.0

    def transform_target(y_values):
        values = np.log1p(y_values) if USE_LOG_TARGET else y_values
        return ((values - target_center) / target_scale).astype("float32")

    def inverse_transform_target(y_scaled):
        values = np.asarray(y_scaled).reshape(-1) * target_scale + target_center
        if USE_LOG_TARGET:
            return np.expm1(values)
        return values

    target_train_base = np.log1p(y_train_np) if USE_LOG_TARGET else y_train_np
    target_center = float(target_train_base.mean())
    target_scale = float(target_train_base.std())
    y_train_model = transform_target(y_train_np)
    y_valid_model = transform_target(y_valid_np)
    y_test_model = transform_target(y_test_np)

    transformed_feature_names = preprocessor.get_feature_names_out()
    split_summary = pd.DataFrame(
        [
            {"Split": "train", "Rows": len(y_train), "MeanPrice": y_train.mean(), "MedianPrice": y_train.median()},
            {"Split": "validation", "Rows": len(y_valid), "MeanPrice": y_valid.mean(), "MedianPrice": y_valid.median()},
            {"Split": "test", "Rows": len(y_test), "MeanPrice": y_test.mean(), "MedianPrice": y_test.median()},
        ]
    )

    target_transform_summary = pd.DataFrame(
        [
            {
                "TrainPriceSkew": target_skew_train,
                "UseLog1pTarget": USE_LOG_TARGET,
                "TargetCenter": target_center,
                "TargetScale": target_scale,
                "RawFeatures": len(feature_cols),
                "TransformedFeatures": X_train.shape[1],
            }
        ]
    )

    print("Rows before duplicate removal:", len(df_raw))
    print("Rows after duplicate removal and valid target filter:", len(df_model))
    display(split_summary.round(2))
    display(target_transform_summary.round(4))
    display(pd.DataFrame({"Feature": transformed_feature_names[:20]}))


Output cho thấy điều gì?

Preprocessing bắt đầu bằng cách bỏ full duplicates, loại `price <= 0` nếu có, và giữ latest sale per property `id` để tránh cùng một property đi qua nhiều split. Split được stratify theo price bins; preprocessor chỉ fit trên train set, còn validation/test chỉ dùng `transform`, nên scaling/encoding không học từ held-out data.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của dữ liệu deep learning và đánh giá CNN suitability cho housing-price tabular regression.


In [ ]:
if RUN_TRAINING:
    representation_summary = pd.DataFrame(
        {
            "Array": ["X_train", "y_train_model", "X_valid", "y_valid_model", "X_test", "y_test_model"],
            "Shape": [X_train.shape, y_train_model.shape, X_valid.shape, y_valid_model.shape, X_test.shape, y_test_model.shape],
            "Meaning": [
                "(samples, transformed features)",
                "(samples,)",
                "(samples, transformed features)",
                "(samples,)",
                "(samples, transformed features)",
                "(samples,)",
            ],
        }
    )
    cnn_suitability = pd.DataFrame(
        [
            {"Question": "Raw representation", "Assessment": "Tabular property vector with numeric and one-hot zipcode features"},
            {"Question": "Natural baseline", "Assessment": "MLP/DNN regression because each house is a fixed feature vector"},
            {"Question": "CNN justified?", "Assessment": "No production CNN: columns do not form an image or meaningful local sequence"},
            {"Question": "CNN usage in this notebook", "Assessment": "Only toy Conv1D operations for understanding convolution, not final modeling"},
        ]
    )
    display(representation_summary)
    display(cnn_suitability)


Output cho thấy điều gì?

`X_train` là ma trận `(samples, transformed features)`, trong đó zipcode đã được one-hot encode và numeric features đã scale. Đây là vector tabular; MLP regression là lựa chọn tự nhiên hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, output regression, Conv1D và pooling bằng toy tensors.


Output cho thấy điều gì?

Regression output là một scalar tuyến tính, không phải xác suất hay class label. Conv1D/pooling chỉ minh họa khi dữ liệu có trục thứ tự; King County sau preprocessing không có cấu trúc cục bộ như vậy.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, forward network, output regression, MSE/MAE helper và Conv1D toy.


In [ ]:
if RUN_TRAINING:
    def linear_np(x, w, b):
        return x @ w + b

    def relu_np(x):
        return np.maximum(0, x)

    def mse_np(y_true, y_pred):
        return np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)

    def mae_np(y_true, y_pred):
        return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

    def forward_regression_np(x, params):
        hidden = relu_np(linear_np(x, params["w1"], params["b1"]))
        return linear_np(hidden, params["w2"], params["b2"]).ravel()

    def conv1d_valid_np(sequence, kernel, bias=0.0):
        sequence = np.asarray(sequence, dtype=float)
        kernel = np.asarray(kernel, dtype=float)
        out = []
        for i in range(len(sequence) - len(kernel) + 1):
            out.append(float(np.sum(sequence[i : i + len(kernel)] * kernel) + bias))
        return np.asarray(out)

    rng = np.random.default_rng(SEED)
    toy_batch = X_train[:4]
    params = {
        "w1": rng.normal(0, 0.1, size=(X_train.shape[1], 8)),
        "b1": np.zeros(8),
        "w2": rng.normal(0, 0.1, size=(8, 1)),
        "b2": np.zeros(1),
    }
    toy_scaled_pred = forward_regression_np(toy_batch, params)
    toy_price_pred = inverse_transform_target(toy_scaled_pred)
    toy_conv = conv1d_valid_np([1, 2, 3, 4, 5], [0.25, 0.5, 0.25])

    print("Toy scaled predictions:", np.round(toy_scaled_pred, 4))
    print("Toy price predictions:", np.round(toy_price_pred, 2))
    print("Toy MSE on scaled target:", round(float(mse_np(y_train_model[:4], toy_scaled_pred)), 4))
    print("Toy MAE on scaled target:", round(float(mae_np(y_train_model[:4], toy_scaled_pred)), 4))
    print("Manual Conv1D toy output:", np.round(toy_conv, 4))


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP regression tạo một giá trị liên tục. MSE/MAE được tính trên target đã transform trong toy example; khi đánh giá model thật, notebook chuyển prediction về price scale gốc.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric regression và baseline Keras MLP với output scalar tuyến tính.


In [ ]:
if RUN_TRAINING:
    def to_jsonable(obj):
        if isinstance(obj, dict):
            return {str(k): to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [to_jsonable(v) for v in obj]
        if isinstance(obj, tuple):
            return [to_jsonable(v) for v in obj]
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    def save_json(path, obj):
        with Path(path).open("w", encoding="utf-8") as f:
            json.dump(to_jsonable(obj), f, indent=2)

    def regression_metrics(y_true_price, y_pred_price):
        y_pred_price = np.maximum(np.asarray(y_pred_price).reshape(-1), 0)
        mse = mean_squared_error(y_true_price, y_pred_price)
        return {
            "MAE": float(mean_absolute_error(y_true_price, y_pred_price)),
            "MSE": float(mse),
            "RMSE": float(np.sqrt(mse)),
            "R2": float(r2_score(y_true_price, y_pred_price)),
        }

    def predict_price_keras(model, X_data):
        scaled_pred = model.predict(X_data, batch_size=BATCH_SIZE, verbose=0).ravel()
        return np.maximum(inverse_transform_target(scaled_pred), 0)

    def compile_regression_model(model):
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss=keras.losses.Huber(delta=1.0),
            metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
        )
        return model

    def build_basic_mlp(input_dim, name="M1_Basic_MLP_Regressor"):
        inputs = keras.Input(shape=(input_dim,), name="property_feature_vector")
        x = layers.Dense(128, activation="relu", name="dense_128")(inputs)
        x = layers.Dense(64, activation="relu", name="dense_64")(x)
        outputs = layers.Dense(1, activation="linear", name="price_output")(x)
        return compile_regression_model(keras.Model(inputs, outputs, name=name))

    BATCH_SIZE = 512
    MAX_EPOCHS = 30
    PATIENCE = 5
    input_dim = X_train.shape[1]

    keras_baseline = build_basic_mlp(input_dim)
    keras_baseline.summary()
    print("Input shape:", X_train.shape)
    print("Output shape for 5 samples:", tuple(keras_baseline(X_train[:5]).shape))
    print("Total parameters:", keras_baseline.count_params())


Output cho thấy điều gì?

Keras baseline nhận vector property đã transform và xuất một scalar linear trên target đã chuẩn hóa. Loss Huber phù hợp với target có skew/outliers vì bớt nhạy với sai số cực lớn hơn MSE thuần.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


Output cho thấy điều gì?

Training log hiển thị loss và MAE trên target đã transform cho train/validation. Checkpoint chọn epoch có validation loss thấp nhất; test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và MAE của baseline Keras trên train/validation, đồng thời lưu figure.


In [ ]:
if RUN_TRAINING:
    def plot_history(history, title, output_path):
        history_df = pd.DataFrame(history.history)
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
        axes[0].plot(history_df['loss'], label='train')
        axes[0].plot(history_df['val_loss'], label='validation')
        axes[0].set_title(f'{title} Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].legend()
        axes[1].plot(history_df['mae'], label='train')
        axes[1].plot(history_df['val_mae'], label='validation')
        axes[1].set_title(f'{title} MAE on transformed target')
        axes[1].set_xlabel('Epoch')
        axes[1].legend()
        fig.tight_layout()
        fig.savefig(output_path, dpi=160)
        plt.show()


Output cho thấy điều gì?

Curves cho biết model học ổn định hay overfit trên validation. Vì target đã log/scale, magnitude của loss dùng để chọn model; metric giá nhà dễ hiểu hơn được tính riêng sau khi inverse transform.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng MAE, MSE, RMSE, R²; lưu actual-vs-predicted, residual plot và error distribution.


Output cho thấy điều gì?

Keras baseline validation đạt MAE 70,219, RMSE 138,387, R² 0.8679. Diagnostic plots cho thấy sai số vẫn lớn hơn ở vùng giá cao, phù hợp với target có đuôi phải dài.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra shape và parameter count.


In [ ]:
if RUN_TRAINING:
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_model.reshape(-1, 1), dtype=torch.float32))
    valid_ds = TensorDataset(torch.tensor(X_valid, dtype=torch.float32), torch.tensor(y_valid_model.reshape(-1, 1), dtype=torch.float32))
    test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test_model.reshape(-1, 1), dtype=torch.float32))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    class TorchMLPRegressor(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 1),
            )

        def forward(self, x):
            return self.net(x)

    def count_torch_params(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    torch_model = TorchMLPRegressor(input_dim).to(DEVICE)
    features_batch, labels_batch = next(iter(train_loader))
    with torch.no_grad():
        output_batch = torch_model(features_batch.to(DEVICE))

    print("Input feature batch shape:", tuple(features_batch.shape))
    print("Regression output batch shape:", tuple(output_batch.shape))
    print("PyTorch parameter count:", count_torch_params(torch_model))


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và output có dạng `(batch, 1)`, đúng với regression scalar. Kiến trúc PyTorch tương đương ý tưởng với Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline bằng loop thủ công, dùng SmoothL1Loss/Huber và lưu checkpoint tốt nhất theo validation loss.


In [ ]:
if RUN_TRAINING:
    def torch_predict_scaled(model, loader):
        model.eval()
        preds = []
        targets = []
        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(DEVICE)
                output = model(xb)
                preds.append(output.cpu().numpy().ravel())
                targets.append(yb.numpy().ravel())
        return np.concatenate(targets), np.concatenate(preds)

    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)

    torch_history_rows = []
    best_val_loss = np.inf
    best_state = None
    epochs_without_improvement = 0
    start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        torch_model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            output = torch_model(xb)
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * xb.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        valid_scaled_true, valid_scaled_pred = torch_predict_scaled(torch_model, valid_loader)
        val_loss = float(np.mean(np.where(np.abs(valid_scaled_pred - valid_scaled_true) < 1, 0.5 * (valid_scaled_pred - valid_scaled_true) ** 2, np.abs(valid_scaled_pred - valid_scaled_true) - 0.5)))
        valid_price_pred_epoch = np.maximum(inverse_transform_target(valid_scaled_pred), 0)
        valid_metrics_epoch = regression_metrics(y_valid_np, valid_price_pred_epoch)
        torch_history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_mae_price": valid_metrics_epoch["MAE"],
                "val_rmse_price": valid_metrics_epoch["RMSE"],
                "val_r2": valid_metrics_epoch["R2"],
            }
        )
        print(f"Epoch {epoch:02d} - train_loss={train_loss:.4f} - val_loss={val_loss:.4f} - val_rmse={valid_metrics_epoch['RMSE']:.2f} - val_r2={valid_metrics_epoch['R2']:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in torch_model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("Early stopping triggered.")
                break

    torch_train_time = time.perf_counter() - start
    torch_model.load_state_dict(best_state)
    torch_checkpoint = MODEL_DIR / "pytorch_basic_mlp_regressor_best.pt"
    torch.save(
        {
            "model_state_dict": torch_model.state_dict(),
            "input_dim": input_dim,
            "feature_columns": feature_cols,
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "use_log_target": USE_LOG_TARGET,
            "target_center": target_center,
            "target_scale": target_scale,
            "seed": SEED,
        },
        torch_checkpoint,
    )
    torch_history = pd.DataFrame(torch_history_rows)
    torch_history.to_csv(METRIC_DIR / "pytorch_basic_mlp_history.csv", index=False)
    print(f"PyTorch train time: {torch_train_time:.2f} seconds")
    display(torch_history.tail())


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, loss, backward và optimizer step. Validation RMSE/R² trong log đã inverse transform về price scale để dễ đọc hơn loss nội bộ.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng metric price-scale như Keras.


In [ ]:
if RUN_TRAINING:
    valid_scaled_true, torch_valid_scaled_pred = torch_predict_scaled(torch_model, valid_loader)
    torch_val_pred = np.maximum(inverse_transform_target(torch_valid_scaled_pred), 0)
    torch_metrics = regression_metrics(y_valid_np, torch_val_pred)
    torch_result = {
        "Model": "M1 Basic MLP Regressor",
        "Framework": "PyTorch",
        **torch_metrics,
        "Params": int(count_torch_params(torch_model)),
        "Train Time": float(torch_train_time),
        "Checkpoint": torch_checkpoint.relative_to(PROJECT_ROOT).as_posix(),
    }
    save_json(METRIC_DIR / "pytorch_basic_mlp_validation_metrics.json", torch_result)

    display(pd.DataFrame([torch_result]).drop(columns=["Checkpoint"]).round(4))


Output cho thấy điều gì?

PyTorch baseline được đánh giá trên cùng validation split và cùng target inverse transform. Metric khác biệt giữa hai framework có thể đến từ initialization và training loop, không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation MAE/RMSE/R².


Output cho thấy điều gì?

Trong baseline framework comparison, PyTorch có validation RMSE thấp nhất (113,519) ở run này. Đây chỉ là kết quả một lần chạy với cùng ý tưởng architecture, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture phù hợp dữ liệu tabular: M1 Basic MLP Regressor, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train như production model vì feature order không có ý nghĩa không gian.


In [ ]:
if RUN_TRAINING:
    def evaluate_keras_model(model, model_key, checkpoint_path, train_time):
        val_pred = predict_price_keras(model, X_valid)
        metrics = regression_metrics(y_valid_np, val_pred)
        result = {'Model': model.name.replace('_', ' '), 'Framework': 'Keras', **metrics, 'Params': int(model.count_params()), 'Train Time': float(train_time), 'Checkpoint': checkpoint_path.relative_to(PROJECT_ROOT).as_posix()}
        save_json(METRIC_DIR / f'{model_key}_validation_metrics.json', result)
        return result


Output cho thấy điều gì?

Các architecture đều giữ cùng split, preprocessing, target transform, seed và training budget. CNN được ghi nhận ở mức methodological investigation, không ép thành model vì bài toán là tabular regression.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation MAE/RMSE/R² và chọn ứng viên tốt nhất bằng validation RMSE thấp nhất.


Output cho thấy điều gì?

`M4 Residual MLP Feature Attention Regressor` là model tốt nhất trên validation với RMSE 119,459, MAE 67,889, R² 0.9015. Sau leakage fix repeated property `id`, M4 attention thắng validation RMSE nhưng vẫn cần cân nhắc vì parameter count cao hơn M1.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


In [ ]:
if RUN_TRAINING:
    hidden_layer = None


Output cho thấy điều gì?

Hidden activations cho thấy model đã biến đổi vector property thành representation phi tuyến trước output giá. Attention weights, nếu dùng M4, là công cụ inspection model chứ không phải bằng chứng nhân quả về thị trường nhà.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation của model được chọn: absolute error lớn nhất, residual theo price range, và nhóm giá thấp/trung/cao.


In [ ]:
if RUN_TRAINING:
    range_rows = []


Output cho thấy điều gì?

Validation error lớn nhất nằm ở nhóm high-price với MAE khoảng 118,650 và RMSE khoảng 189,305. Điều này phù hợp với phân phối giá lệch phải: nhà giá cao tạo residual lớn hơn và làm RMSE nhạy hơn MAE.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation RMSE, evaluate test set đúng một lần, rồi lưu final model và diagnostics.


Output cho thấy điều gì?

Final test evaluation của `M4 Residual MLP Feature Attention Regressor` đạt MAE 70,282, RMSE 146,275, R² 0.8673 trên price scale gốc sau inverse transform. Test set chỉ dùng ở bước này sau khi chọn model bằng validation RMSE.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không dùng score difference như benchmark trực tiếp.


Output cho thấy điều gì?

A03 và A04 cùng là housing-price regression nhưng khác thị trường, target scale và feature schema. Vì vậy chênh lệch RMSE/R² không chứng minh model A04 tốt hơn hay kém hơn A03; nó chỉ mô tả kết quả trên dataset A04.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


In [ ]:
if RUN_TRAINING:
    artifact_listing = []


Output cho thấy điều gì?

Notebook đã lưu final model, preprocessor, comparison metrics, error-analysis CSV và figures. Kết quả cuối: selected model là `M4 Residual MLP Feature Attention Regressor`, test MAE 70,282, RMSE 146,275, R² 0.8673; tổng training/evaluation time được ghi nhận khoảng 82.39 giây.


In [ ]:
from IPython.display import display
import pandas as pd
_saved = _repo / 'A04/artifacts/pipeline04/house_price/metrics'
for _file in sorted(_saved.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))
